# Conformal Triage — Fase 3: splits, head, capa conformal y evaluacion

Corre sobre los embeddings que ya estan en tu Drive (`conformal-triage/emb/`). **No hace falta GPU**:
entorno de ejecucion CPU alcanza. Tarda ~10–15 min (casi todo es el head). `SEED = 2026` es el seed
que queda registrado en el paper — si queres otro, cambialo antes de esta corrida y nunca mas.

Al final deja todo en `conformal-triage/results/`: `resumen_seccion5.csv` (Tablas 7-8 del paper),
`table6_final.csv` (Tabla 6) y `head_report.txt`.


In [ ]:
# 1) Setup: Drive + embeddings + metadata (baja los 2 CSV que faltaron y los guarda en tu Drive)
SEED = 2026   # seed final del paper: no lo cambies despues de la corrida real
import numpy as np, pandas as pd, os, json, requests
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/conformal-triage'
os.makedirs(f'{BASE}/results', exist_ok=True)

d = np.load(f'{BASE}/emb/hiba_emb.npz');  Eh = dict(zip(d['ids'], d['features'].astype(np.float32)))
d = np.load(f'{BASE}/emb/pad_emb.npz');   Ep = dict(zip(d['ids'], d['features'].astype(np.float32)))
Xi, yi = [], []
for k in range(3):
    d = np.load(f'{BASE}/emb/isic2019_emb_part{k}.npz')
    Xi.append(d['features'].astype(np.float32)); yi.append(d['labels'])
    assert list(d['classes']) == ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
Xisic = np.concatenate(Xi); yisic = np.concatenate(yi).astype(int)
print('embeddings:', len(Eh), 'hiba |', len(Ep), 'pad |', Xisic.shape, 'isic2019')

def get_meta(cid, name, esperado):
    p = f'{BASE}/emb/{name}'
    if not os.path.exists(p):
        r = requests.get(f'https://api.isic-archive.com/collections/{cid}/metadata/', timeout=180)
        r.raise_for_status(); open(p, 'wb').write(r.content)
    m = pd.read_csv(p)
    assert len(m) == esperado, f'{name}: {len(m)} filas, esperaba {esperado}. Verifica el id con: isic collection list'
    return m
# ids verificados contra el archive (2026-08-20): HIBA = 251, PAD-UFES-20 (mirror oficial) = 406
Hm = get_meta(251, 'hiba_isic_metadata.csv', 1616)
Pm = get_meta(406, 'pad_isic_metadata.csv', 2298)
print('metadata OK:', len(Hm), 'hiba |', len(Pm), 'pad')


In [ ]:
# 2) Funciones del pipeline (armonizacion, splits, de-clustering, APS, Mondrian, triage)
import numpy as np, pandas as pd

CLS = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
MAL = {'MEL','BCC','SCC','AK'}
ROMAN = {'I':1,'II':2,'III':3,'IV':4,'V':5,'VI':6}
HIBA_MAP = {'melanoma':'MEL','nevus':'NV','basal cell carcinoma':'BCC',
            'squamous cell carcinoma':'SCC','actinic keratosis':'AK',
            'seborrheic keratosis':'BKL','solar lentigo':'BKL',
            'lichenoid keratosis':'BKL','dermatofibroma':'DF','vascular lesion':'VASC'}
DX3_MAP = {'Melanoma, NOS':'MEL','Melanoma':'MEL','Nevus':'NV',
           'Basal cell carcinoma':'BCC','Squamous cell carcinoma, NOS':'SCC',
           'Squamous cell carcinoma':'SCC','Solar or actinic keratosis':'AK',
           'Seborrheic keratosis':'BKL','Solar lentigo':'BKL',
           'Lichen planus like keratosis':'BKL','Lichenoid keratosis':'BKL',
           'Dermatofibroma':'DF','Hemangioma':'VASC','Angioma':'VASC',
           'Vascular lesion':'VASC',
           'Benign soft tissue proliferations - Vascular':'VASC'}

def lesion_table(meta, cohort):
    """Una fila por lesion: cls, patient, g (fototipo 1..6 o NaN), lista de isic_id."""
    df = meta.copy()
    if 'diagnosis' in df.columns and df['diagnosis'].notna().any():
        dx = df['diagnosis']            # esquema viejo del archive (export antiguo)
    else:                               # esquema nuevo: diagnosis_1/2/3; vasculares
        dx = df['diagnosis_3'].fillna(df['diagnosis_2'])  # viven en diagnosis_2
    df['cls'] = dx.map({**HIBA_MAP, **DX3_MAP})
    df['g'] = df['fitzpatrick_skin_type'].map(ROMAN)
    faltan = sorted(dx[df['cls'].isna()].dropna().unique().tolist()) + (['(vacio)'] if (df['cls'].isna() & dx.isna()).any() else [])
    assert not faltan, f'{cohort}: diagnosticos sin mapear: {faltan} (agregalos al mapa)'
    les = df.groupby('lesion_id').agg(cls=('cls','first'), patient=('patient_id','first'),
                                      g=('g','first'), imgs=('isic_id', lambda s: tuple(s))).reset_index()
    chk = df.groupby('lesion_id').agg(nc=('cls','nunique'), npat=('patient_id','nunique'))
    assert (chk.nc == 1).all() and (chk.npat == 1).all(), f'{cohort}: lesion con clase o paciente inconsistente'
    les['mal'] = les.cls.isin(MAL)
    return les

def membership(les):
    """patient -> set de estratos: ('M', g) por lesion maligna fototipada; ('B',) por benigna fototipada."""
    m = {}
    ph = les[les.g.notna()]
    for pid, sub in ph.groupby('patient'):
        s = set()
        for _, r in sub.iterrows():
            s.add(('M', int(r.g)) if r.mal else ('B',))
        m[pid] = s
    return m

def floor_n(alpha):
    return int(np.ceil(1.0/alpha - 1e-9)) - 1

def pad_training_draw(les, rng, train_frac=0.4, cal_frac=0.6, alpha=0.05, max_tries=100):
    """40% de pacientes a training, estratificado por fototipo (incl. grupo sin fototipo),
    verificado: cada estrato obligatorio retiene en el remanente pacientes suficientes para
    que la fraccion de calibracion supere el piso en esperanza. Se redibuja si falla."""
    pf = les.groupby('patient')['g'].agg(lambda s: tuple(s.dropna().unique()))
    pgroup = {p: (int(v[0]) if len(v) else 0) for p, v in pf.items()}
    pats = np.array(sorted(pgroup)); grp = np.array([pgroup[p] for p in pats])
    mem = membership(les)
    enforced = [('M',1), ('M',2), ('M',3), ('B',)]
    fl = floor_n(alpha)
    for _ in range(max_tries):
        tr = set()
        for g in np.unique(grp):
            idx = np.where(grp == g)[0]
            tr.update(pats[rng.choice(idx, int(round(train_frac*len(idx))), replace=False)])
        rem = [p for p in pats if p not in tr]
        ok = all(cal_frac*sum(1 for p in rem if s in mem.get(p, ())) >= fl for s in enforced)
        if ok:
            return sorted(tr), sorted(rem)
    raise RuntimeError('no se pudo verificar el training draw')

def aps_scores(P):
    """S[i, c] = suma de probabilidades >= p_c (inclusive), APS no aleatorizado."""
    order = np.argsort(-P, axis=1)
    cum = np.take_along_axis(P, order, 1).cumsum(1)
    S = np.empty_like(P)
    np.put_along_axis(S, order, cum, 1)
    return S

def qhat(scores, alpha):
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha) - 1e-9))
    if n == 0 or k > n:
        return np.inf
    return np.sort(scores)[k - 1]

def decluster(les_cal, mem_cal, rng):
    """Por estrato, una lesion por paciente (uniforme entre sus lesiones del estrato).
    Devuelve dict estrato -> indices de lesiones (posiciones en les_cal)."""
    out = {}
    ph = les_cal[les_cal.g.notna()]
    for s in set().union(*mem_cal.values()) if mem_cal else set():
        rows = ph[(ph.mal & (ph.g == s[1]))] if s[0] == 'M' else ph[~ph.mal]
        picks = [rng.choice(sub.index.values) for _, sub in rows.groupby('patient')]
        out[s] = np.array(sorted(picks), dtype=int)
    return out

def evaluate_draw(les, S, cal_pats, test_pats, alphas, rng):
    """S: matriz de scores APS alineada al indice de `les`. Devuelve registros por (alpha, estrato)."""
    cal = les[les.patient.isin(cal_pats)]
    test = les[les.patient.isin(test_pats) & les.g.notna()]
    mem_cal = membership(cal)
    cells = decluster(cal, mem_cal, rng)
    strata = sorted(cells, key=str)
    recs = []
    for a in alphas:
        q = {}
        for s in strata:
            idx = cells[s]
            q[s] = qhat(S[idx, [CLS.index(les.loc[i, 'cls']) for i in idx]], a) if len(idx) else np.inf
        qB = q.get(('B',), np.inf)
        Ct = np.zeros((len(test), len(CLS)), bool)
        Stest = S[test.index.values]
        for c, name in enumerate(CLS):
            if name in MAL:
                thr = np.array([q.get(('M', int(g)), np.inf) for g in test.g.values])
            else:
                thr = np.full(len(test), qB)
            Ct[:, c] = Stest[:, c] <= thr
        ytrue = np.array([CLS.index(c) for c in test.cls.values])
        covered = Ct[np.arange(len(test)), ytrue]
        single_benign = (Ct.sum(1) == 1) & ~Ct[:, [CLS.index(c) for c in CLS if c in MAL]].any(1)
        observe = single_benign
        for s in strata:
            n_cal = len(cells[s])
            if s[0] == 'M':
                sel = test.mal.values & (test.g.values == s[1])
            else:
                sel = ~test.mal.values
            if sel.sum() == 0 and n_cal == 0:
                continue
            miss = float((observe[sel] & test.mal.values[sel]).mean()) if s[0] == 'M' and sel.sum() else np.nan
            if sel.sum():
                pw = pd.DataFrame({'p': test.patient.values[sel], 'cov': covered[sel],
                                   'ref': ~observe[sel]}).groupby('p').mean().mean()
            recs.append(dict(alpha=a, stratum=f'{s[0]}{s[1] if s[0]=="M" else ""}',
                             n_cal=n_cal, degenerate=bool(np.isinf(q[s])),
                             coverage=float(covered[sel].mean()) if sel.sum() else np.nan,
                             coverage_pw=float(pw['cov']) if sel.sum() else np.nan,
                             miss_rate=miss,
                             referral=float((~observe[sel]).mean()) if sel.sum() else np.nan,
                             referral_pw=float(pw['ref']) if sel.sum() else np.nan,
                             set_size=float(Ct[sel].sum(1).mean()) if sel.sum() else np.nan,
                             n_test=int(sel.sum()), n_test_pat=int(test.patient.values[sel].size and len(set(test.patient.values[sel])))))
        # baseline marginal: un solo umbral global (de-cluster global: una lesion por paciente)
        picks = [rng.choice(sub.index.values) for _, sub in cal[cal.g.notna()].groupby('patient')]
        picks = np.array(sorted(picks), dtype=int)
        qm = qhat(S[picks, [CLS.index(les.loc[i, 'cls']) for i in picks]], a)
        Cm = Stest <= qm
        obs_m = (Cm.sum(1) == 1) & ~Cm[:, [CLS.index(c) for c in CLS if c in MAL]].any(1)
        for g in sorted(test.g.dropna().unique()):
            sel = test.mal.values & (test.g.values == g)
            if sel.sum():
                recs.append(dict(alpha=a, stratum=f'marginal-M{int(g)}', n_cal=len(picks),
                                 degenerate=bool(np.isinf(qm)),
                                 coverage=float(Cm[np.arange(len(test)), ytrue][sel].mean()),
                                 miss_rate=float((obs_m[sel]).mean()),
                                 referral=float((~obs_m[sel]).mean()),
                                 set_size=float(Cm[sel].sum(1).mean()), n_test=int(sel.sum())))
    return recs



In [ ]:
# 3) Armonizacion + verificacion contra la Tabla 5 del paper (PASS/FAIL)
hles = lesion_table(Hm, 'hiba').reset_index(drop=True)
ples = lesion_table(Pm, 'pad').reset_index(drop=True)

def check(nombre, real, esperado):
    ok = real == esperado
    print(f"{'PASS' if ok else 'FAIL'}  {nombre}: {real}" + ('' if ok else f' (esperaba {esperado})'))
    return ok

ok = True
ok &= check('HIBA lesiones/pacientes/fototipadas', (len(hles), hles.patient.nunique(), int(hles.g.notna().sum())), (1246, 623, 1139))
mI, mIII = hles[hles.mal & (hles.g==1)], hles[hles.mal & (hles.g==3)]
ok &= check('HIBA M-I lesiones/pacientes', (len(mI), mI.patient.nunique()), (59, 34))
ok &= check('HIBA M-III lesiones/pacientes', (len(mIII), mIII.patient.nunique()), (44, 34))
mel2 = hles[(hles.cls=='MEL') & (hles.g==2)]
ok &= check('HIBA MEL tipo II lesiones/pacientes', (len(mel2), mel2.patient.nunique()), (125, 119))
hm = membership(hles)
ok &= check('HIBA pacientes M/B', (sum(1 for s in hm.values() if any(x[0]=='M' for x in s)), sum(1 for s in hm.values() if ('B',) in s)), (383, 238))
ok &= check('PAD lesiones/pacientes/fototipadas', (len(ples), ples.patient.nunique(), int(ples.g.notna().sum())), (1891, 1373, 1179))
T5 = {1: (2,85,12,21,3,4,70,6), 2: (24,386,95,143,27,19,445,42), 3: (8,164,33,72,24,5,197,21),
      4: (2,15,3,9,8,5,26,11), 5: (0,2,2,3,1,1,6,2), 6: (0,0,0,0,0,1,0,1)}
pm = membership(ples)
for g, exp in T5.items():
    sub = ples[ples.g == g]
    cnt = tuple(int((sub.cls==c).sum()) for c in ['MEL','BCC','SCC','AK','NV','BKL'])
    Mp = sum(1 for s in pm.values() if ('M', g) in s)
    Bp_g = sub[~sub.mal].patient.nunique()
    ok &= check(f'PAD Tabla 5 fototipo {g} (MEL,BCC,SCC,AK,NV,BKL,Mpac,Bpac)', cnt + (Mp, Bp_g), exp)
ben = ples[ples.g.notna() & ~ples.mal]
ok &= check('PAD benigno agrupado lesiones/pacientes', (len(ben), ben.patient.nunique()), (98, 83))
faltan = [i for t in list(hles.imgs) for i in t if i not in Eh] + [i for t in list(ples.imgs) for i in t if i not in Ep]
ok &= check('imagenes sin embedding', len(faltan), 0)
assert ok, 'FALLO la verificacion: revisa las lineas FAIL antes de seguir'
print('\nVerificacion completa: las dos publicaciones del dataset coinciden con el paper.')


In [ ]:
# 4) Training draw de PAD (estratificado + verificado, seed fijo) y entrenamiento del head
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, classification_report

rng = np.random.default_rng(SEED)
tr, rem = pad_training_draw(ples, rng)
print(f'PAD: {len(tr)} pacientes a training, {len(rem)} al remanente calibracion/test')

pad_tr = ples[ples.patient.isin(tr)]
Xpt = np.stack([Ep[i] for t in pad_tr.imgs for i in t])
ypt = np.array([CLS.index(r.cls) for _, r in pad_tr.iterrows() for _ in r.imgs])
X = np.vstack([Xisic, Xpt]); y = np.concatenate([yisic, ypt])
sc = StandardScaler().fit(X)
head = LogisticRegression(max_iter=1000, class_weight='balanced').fit(sc.transform(X), y)
rep = classification_report(y, head.predict(sc.transform(X)), target_names=CLS, digits=3)
bal = balanced_accuracy_score(y, head.predict(sc.transform(X)))
print(f'head entrenado sobre {len(y)} imagenes | balanced accuracy (train): {bal:.3f}')
open(f'{BASE}/results/head_report.txt', 'w').write(
    f'seed={SEED}\ntrain: ISIC 2019 completo + PAD training ({len(tr)} pacientes)\n'
    f'imagenes de entrenamiento: {len(y)}\nbalanced accuracy (train): {bal:.4f}\n\n{rep}')
np.savez_compressed(f'{BASE}/results/head_weights.npz', coef=head.coef_, intercept=head.intercept_,
                    scaler_mean=sc.mean_, scaler_scale=sc.scale_, classes=np.array(CLS), seed=SEED)

def lesion_probs(les, E):
    Xl = np.stack([np.mean([E[i] for i in t], 0) for t in les.imgs])
    return head.predict_proba(sc.transform(Xl))
Sh = aps_scores(lesion_probs(hles, Eh))
Sp = aps_scores(lesion_probs(ples, Ep))
print('scores APS listos:', Sh.shape, Sp.shape)


In [ ]:
# 5) Evaluacion pre-registrada: 100 particiones patient-level, alfa = 0.05 / 0.10 / 0.15
ALPHAS = [0.05, 0.10, 0.15]
hp = sorted(membership(hles))
allrec = []
for d in range(100):
    r = np.random.default_rng(SEED * 1000 + d)
    cal = set(r.choice(hp, int(round(0.7 * len(hp))), replace=False))
    allrec += [dict(cohort='hiba', draw=d, **x) for x in evaluate_draw(hles, Sh, cal, set(hp) - cal, ALPHAS, r)]
    calp = set(r.choice(rem, int(round(0.6 * len(rem))), replace=False))
    allrec += [dict(cohort='pad', draw=d, **x) for x in evaluate_draw(ples, Sp, calp, set(rem) - calp, ALPHAS, r)]
    if d % 20 == 0: print(f'  draw {d}/100')
R = pd.DataFrame(allrec)
R.to_csv(f'{BASE}/results/results_draws.csv', index=False)
res = R.groupby(['cohort', 'stratum', 'alpha']).agg(
    n_cal=('n_cal', 'mean'), deg_rate=('degenerate', 'mean'),
    cov=('coverage', 'mean'), cov_sd=('coverage', 'std'), cov_min=('coverage', 'min'),
    cov_pw=('coverage_pw', 'mean'), miss=('miss_rate', 'mean'), miss_sd=('miss_rate', 'std'),
    ref=('referral', 'mean'), ref_pw=('referral_pw', 'mean'),
    set_size=('set_size', 'mean'), n_test=('n_test', 'mean')).round(4)
res.to_csv(f'{BASE}/results/resumen_seccion5.csv')
print('\n=== resumen alfa = 0.05 (estratos del esquema primario) ===')
v = res.xs(0.05, level='alpha')
print(v[~v.index.get_level_values('stratum').str.startswith('marginal')].to_string())


In [ ]:
# 6) Tabla 6 final: auditoria de certificabilidad con el seed del paper (4.000 draws)
from scipy.stats import beta as Beta
rngA = np.random.default_rng(SEED)
def floor_a(a): return int(np.ceil(1.0/a - 1e-9)) - 1

def audit(les, draws=4000, cohort='hiba'):
    mem = membership(les)
    strata = sorted(set().union(*mem.values()), key=str)
    counts = {s: [] for s in strata}
    if cohort == 'hiba':
        pats = np.array(sorted(mem))
        for _ in range(draws):
            cal = set(rngA.choice(pats, int(round(0.7 * len(pats))), replace=False))
            for s in strata: counts[s].append(sum(1 for p in cal if s in mem[p]))
    else:
        pf = les.groupby('patient')['g'].agg(lambda s: tuple(s.dropna().unique()))
        pg = {p: (int(v[0]) if len(v) else 0) for p, v in pf.items()}
        pats = np.array(sorted(pg)); grp = np.array([pg[p] for p in pats])
        for _ in range(draws):
            t = set()
            for g in np.unique(grp):
                idx = np.where(grp == g)[0]
                t.update(pats[rngA.choice(idx, int(round(0.4 * len(idx))), replace=False)])
            rm = np.array([p for p in pats if p not in t])
            cal = set(rngA.choice(rm, int(round(0.6 * len(rm))), replace=False))
            for s in strata: counts[s].append(sum(1 for p in cal if s in mem.get(p, ())))
    rows = []
    for s in strata:
        v = np.array(counts[s]); n = int(round(v.mean()))
        l = int(np.floor((n + 1) * 0.05 + 1e-9))
        cov5 = Beta.ppf(0.05, n + 1 - l, l) if l >= 1 else np.nan
        rows.append(dict(cohort=cohort, stratum=f"{s[0]}{s[1] if s[0]=='M' else ''}",
                         ncal_mean=round(v.mean(), 1), ncal_min=int(v.min()),
                         alpha_min=round(1 / (v.mean() + 1), 3),
                         cov5=round(cov5, 3) if l >= 1 else None,
                         deg05=round(100 * (v < floor_a(0.05)).mean(), 2),
                         deg10=round(100 * (v < floor_a(0.10)).mean(), 2),
                         deg15=round(100 * (v < floor_a(0.15)).mean(), 2)))
    return rows

T6 = pd.DataFrame(audit(hles, cohort='hiba') + audit(ples, cohort='pad'))
T6.to_csv(f'{BASE}/results/table6_final.csv', index=False)
print(T6.to_string(index=False))
print('\nListo. Salidas en Drive/conformal-triage/results/.')
